In [1]:
cd /Users/karolinegriesbach/Documents/Innkeepr/Git/evaluation-and-execution-scripts

In [2]:
import pandas as pd
import delta_sharing

import general_functions.databricks_client as db_client
from general_functions.return_workspace_ids import return_workspace_ids
from general_functions.constants import return_api_url

In [3]:
customer = "LILLYDOO"
url = return_api_url()
print(f"url = {url}")
workspace_id = return_workspace_ids()
workspace_id = [acc["id"] for acc in workspace_id if acc["name"] == customer]
workspace_id = workspace_id[0]

In [4]:
profile_path = db_client.return_databricks_client()
table_name = "bigquery_adtribute"
table_path = f"{profile_path}#delta_share_events.{workspace_id}.{table_name}"
df = delta_sharing.load_as_pandas(table_path)#, limit=20000)

In [5]:
df.shape

In [6]:
min_date = df["date"].min()
max_date = df["date"].max()
print(f"session daterange = {min_date} - {max_date}")
min_date_bq = df["bigquery_created"].min()
max_date_bq = df["bigquery_created"].max()
print(f"bigquery daterange = {min_date_bq} - {max_date_bq}")

In [7]:
df["bigquery_date_ymd"] = pd.to_datetime(df["bigquery_created"]).dt.date
df.groupby("bigquery_date_ymd")["anonymousId"].count().sort_values(ascending=False).head(10)

In [8]:
df.groupby("date")["anonymousId"].nunique().sort_index(ascending=False)

In [9]:
df.columns

In [10]:
matched_anonymous_ids = df[df["anonymousId"].notnull()]["bigquery_conversion_id"].nunique()
unmatched_anonymous_ids = df[df["anonymousId"].isnull()]["bigquery_conversion_id"].nunique()

print(f"matched_anonymous_ids = {matched_anonymous_ids}")
print(f"unmatched_anonymous_ids = {unmatched_anonymous_ids}")
matching_rate = matched_anonymous_ids/(matched_anonymous_ids+unmatched_anonymous_ids)*100
print(f"matching_rate = {matching_rate}")

In [11]:
df.isnull().sum()

# Check with profiles

In [12]:
profiles = delta_sharing.load_as_pandas(f"{profile_path}#delta_share_events.{workspace_id}.profiles")#, limit=20000)

In [13]:
unmatched_emails = df[df["anonymousId"].isnull()]["email_sha256"].unique()
print(f"unmatched_emails = {len(unmatched_emails)}")

In [14]:
import ast
def return_email_sha256(x):
    try:
        if x == "None" or x is None:
            return None
        elif pd.notnull(x):
            if isinstance(x, str):
                x = ast.literal_eval(x)
            return [entry.get("id",None) for entry in x]
        else:
            raise Exception(f"x = {x}: {type(x)} cannot be processed")
    except Exception as e:
        raise Exception(f"x = {x}: {e}")

emails = profiles[["anonymousId","email_sha256_externalIds"]]
emails["email"] = emails["email_sha256_externalIds"].apply(lambda x: return_email_sha256(x))
emails = emails.explode("email")
emails

In [15]:
emails[emails["email"].isin(unmatched_emails)]